In [ ]:
from unittest import result

import pandas as pd

import matplotlib.pyplot as plt
import re


from src.readability_filter import readability_drop, readable
from src.faithfulness_filter import is_faithful

In [ ]:
import json
#loading the data
with open("../data/CUADv1.json") as f:
    data = json.load(f)

test = pd.DataFrame.from_records(data)

test.info()


In [ ]:
test.describe()


In [ ]:
#extracting and reorganizing clauses
rows = []
for contract in data["data"]:
    title = contract["title"]
    for qa in contract["paragraphs"][0]["qas"]:
        if qa["is_impossible"]:
            continue
        category = qa["question"].split('"')[1]
        for answer in qa["answers"]:
            rows.append({"contract": title, "category": category, "clause_text": answer["text"].strip()})

df = pd.DataFrame(rows)
df.head()

freqfreq{'title': 'LIMEENERGYCO_09_09_1999-…Data: list of contracts
contract: {title, paragraph}
title: contract name
paragraph: list with one entry
-context : text of contract
-qas : list of 41 questions
-- question:
-- id
-- is_imossible (if contract does not have clause)
-- answers :
    -answer
        -text: clause
        -answer_start : index of clasue in context


In [ ]:
df["category"].value_counts().plot(kind="barh", figsize=(8, 12))
plt.show()

In [ ]:
df["word_count"] = df["clause_text"].str.split().str.len()
df["word_count"].hist(bins=50)
category_words = df.groupby("category")["word_count"].agg(["count"])

plt.xlabel("words per clause")
plt.ylabel("count")
plt.show()

In [ ]:
category_stats = df.groupby("category")["word_count"].agg(["count", "median", "min", "max"]).sort_values("median")

category_stats.head(10)


In [ ]:
df.boxplot(column="word_count", by="category", vert=False, figsize=(10, 14))
plt.suptitle("")
plt.title("Words per category")
plt.xlabel("num words")
plt.tight_layout()
plt.show()

In [ ]:
Delete = ["Parties", "Document Name", "Agreement Date", "Effective Date"]
df_filtered = df[~df["category"].isin(Delete)]

In [ ]:
df_filtered["category"].value_counts().plot(kind="barh", figsize=(8, 12))
plt.show()

In [ ]:
df_filtered.sort_values("word_count")
df_filtered["clause_text"].str.contains(r"\[\*+\]", regex=True).sum()


In [ ]:
df_filtered.sort_values("word_count")
df_wordcount8 =df_filtered[df_filtered["word_count"] >= 8]
df_wordcount8.sort_values("word_count").head(30)




In [ ]:
df_clean = df_wordcount8.copy()
df_clean["clause_text"] = df_clean["clause_text"].str.replace("\xa0", " ", regex=False)

In [ ]:
AMOUNT_CATEGORIES = {"Minimum Commitment", "Revenue/Profit Sharing", "Cap On Liability",
                     "Liquidated Damages", "Price Restrictions", "Volume Restriction", "Insurance"}
DURATION_CATEGORIES = {"Post-Termination Services", "Warranty Duration",
                       "Renewal Term", "Notice Period To Terminate Renewal"}


def replace_redaction(row):
    if row["category"] in AMOUNT_CATEGORIES:
        placeholder = "[REDACTED AMOUNT]"
    elif row["category"] in DURATION_CATEGORIES:
        placeholder = "[REDACTED DURATION]"
    else:
        placeholder = "[REDACTED]"
    return re.sub(r"\[\*+\]", placeholder, row["clause_text"])

In [ ]:
df_clean["clause_text"] = df_clean.apply(replace_redaction, axis=1)

print(df_clean["clause_text"].str.contains("[REDACTED", regex=False).sum())
print(df_clean["clause_text"].str.contains(r"\[\*+\]", regex=True).sum())

In [ ]:
df_clean.to_csv("../data/cuad_clauses.csv")


In [ ]:
df_clean = pd.read_csv("../data/cuad_clauses.csv")

In [ ]:
df_sample = df_clean.sample(frac=1, random_state=1).groupby("category").head(60)


In [ ]:
print(len(df_sample))
print(df_sample["category"].nunique())
df_sample["category"].value_counts()

In [ ]:
df_sample.to_csv("../data/cuad_60category_sampled.csv", index=False)

In [ ]:
df_sample = pd.read_csv("../data/cuad_60category_sampled.csv")
df_sample


In [ ]:

#taking one contract per category, so that every category is present in the test set
test_contracts = (df_sample.sample(frac=1, random_state=1).groupby("category").head(1))
test_contracts = test_contracts["contract"].unique()

clauses_in_test_contracts = df_sample["contract"].isin(test_contracts)
test_cuad = df_sample[clauses_in_test_contracts]
training_cuad = df_sample[~clauses_in_test_contracts]

contracts_in_both = test_cuad[test_cuad["contract"].isin(training_cuad["contract"])]


print(len(test_cuad), len(training_cuad))
print(contracts_in_both.info())
print(test_cuad["category"].nunique())


In [ ]:
duplicates = training_cuad["clause_text"].isin(test_cuad["clause_text"])
training_cuad = training_cuad[~duplicates]
print(duplicates.sum())

In [ ]:
#saving
test_cuad.to_csv("../data/cuad_test.csv", index=False)
training_cuad.to_csv("../data/cuad_train.csv", index=False)